In [0]:
"""
03_production_kpis.py

Manufacturing Production KPIs

Source:
    fact_production

Target:
    production_kpis

Author:
Sumanth Vempalle

Version:
2.0.0
"""

import dlt

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    current_timestamp,
    sum,
)

# ============================================================
# Production KPIs
# ============================================================

@dlt.table(
    name="production_kpis",
    comment="Manufacturing production KPIs.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)
def production_kpis():

    production = dlt.read("fact_production")

    return (

        production

        .groupBy(

            "plant_code",
            "product_code",
            "product_name",
            "family",
            "planned_shift",

        )

        .agg(

            countDistinct(
                "work_order_id"
            ).alias(
                "work_orders_created"
            ),

            countDistinct(
                "execution_id"
            ).alias(
                "executions_started"
            ),

            sum(
                "quantity"
            ).alias(
                "products_started"
            ),

            countDistinct(
                "sap_order_number"
            ).alias(
                "sap_orders"
            ),

        )

        .withColumn(

            "generated_timestamp",

            current_timestamp()

        )

    )